# Lorenz, true equations: entropy transfers

Pires, Docquier, Vannitsem (2024), arXiv 2307.10623v2. Lorenz (1963), ref. [26].

## Parameters

Lorenz constants and scaling (Sec. 6); `dt` (Sec. 6.2).

In [1]:
import random, torch
torch.set_default_dtype(torch.float64)

sig, Ra, beta = 10.0, 27.0, 8 / 3
mu3, s1, s2, s3 = 22.61, 7.74, 8.76, 8.35
dt = 0.02

## Equations (Eq. 71a–c, noise off)

$$
\frac{dX_1}{dt} = \sigma\frac{s_2}{s_1}X_2 - \sigma X_1
$$

$$
\frac{dX_2}{dt} = \frac{s_1}{s_2}(R_a - \mu_3)X_1 - \frac{s_1 s_3}{s_2}X_1X_3 - X_2
$$

$$
\frac{dX_3}{dt} = \frac{s_1 s_2}{s_3}X_1X_2 - \beta X_3 - \beta\frac{\mu_3}{s_3}
$$

In [2]:
def F(x1, x2, x3):
    return [
        (sig * s2 / s1) * x2 - sig * x1,
        (s1 / s2) * (Ra - mu3) * x1 - (s1 * s3 / s2) * x1 * x3 - x2,
        (s1 * s2 / s3) * x1 * x2 - beta * x3 - beta * mu3 / s3,
    ]

## Simulate (App. B2, Eq. B.6–B.8)

$$
X^* = X + dt\,F(X), \qquad X^{**} = X + dt\,F(X^*), \qquad X_{\text{new}} = \tfrac12\,(X^* + X^{**})
$$

Drop the first 100 time units, keep 400 (Sec. 6.2). One run, random start (paper: 30 runs).

In [3]:
random.seed(0)
X = [random.gauss(0, 1) for _ in range(3)]
run = []
for t in range(int(500 / dt)):
    f = F(*X)
    Xs = [X[i] + dt * f[i] for i in range(3)]          # B.6
    fs = F(*Xs)
    Xss = [X[i] + dt * fs[i] for i in range(3)]        # B.7
    X = [0.5 * (Xs[i] + Xss[i]) for i in range(3)]     # B.8
    if t >= int(100 / dt):
        run.append(X)

torch.save(run, "lorenz_run.pt")

In [4]:
X1 = [p[0] for p in run]
X2 = [p[1] for p in run]
X3 = [p[2] for p in run]
N = len(run)
print(N, "points")

20000 points


## Tool 1: `fit_curve(x, y)`

Degree-10 polynomial through the cloud: the curve $E(y \mid x)$ (Sec. 4.2; $N_d = 10$, Sec. 6.2).

Dummy: points on $1 + 2x + 3x^2$, degree 2 → `[1, 2, 3]`.

In [5]:
def fit_curve(x, y, deg=10):
    A = [[xn ** p for p in range(deg + 1)] for xn in x]
    Y = [[yn] for yn in y]
    return torch.linalg.lstsq(torch.tensor(A), torch.tensor(Y)).solution[:, 0].tolist()

In [6]:
xs = [0.0, 1.0, 2.0, 3.0]
print(fit_curve(xs, [1 + 2 * x + 3 * x ** 2 for x in xs], deg=2))

[0.9999999999999998, 2.0000000000000013, 2.9999999999999996]


## Tool 2: `value(c, x)`, `slope(c, x)`

Height and derivative of the curve at `x`.

Dummy: `c = [1, 2, 3]` at $x = 2$ → 17, 14.

In [7]:
def value(c, x):
    return sum(c[p] * x ** p for p in range(len(c)))

def slope(c, x):
    return sum(p * c[p] * x ** (p - 1) for p in range(1, len(c)))

In [8]:
print(value([1, 2, 3], 2), slope([1, 2, 3], 2))

17 14


## All $\to X_3$ (Eq. 11, Sec. 2.2)

$$
T(X_{\sim 3}\to X_3) = E\Big[\frac{d}{dX_3}E\big(F_{3,\sim 3}\mid X_3\big)\Big] - E\Big[\frac{\partial F_{3,\sim 3}}{\partial X_3}\Big]
$$

$F_{3,\sim 3}$ = terms of $F_3$ with $X_1$ or $X_2$:

$$
F_{3,\sim 3} = \frac{s_1 s_2}{s_3}X_1X_2
$$

In [9]:
F3_cross = [(s1 * s2 / s3) * X1[n] * X2[n] for n in range(N)]

First term: curve against $X_3$ → slope at each point → average.

In [10]:
c = fit_curve(X3, F3_cross)

first = 0.0
for n in range(N):
    first += slope(c, X3[n])
first = first / N
print("first term:", first)

first term: 2.6330607599837537


Second term: 0 ($F_{3,\sim 3}$ has no $X_3$).

Paper: 2.63 (Table 3, single + synergetic).

In [11]:
second = 0.0
print("T(X1,X2 -> X3):", first - second, "  (paper: 2.63)")

T(X1,X2 -> X3): 2.6330607599837537   (paper: 2.63)


## One $\to X_2$ (Eq. 37a, Corollary 4, p. 14)

For a term $F_{i,j} = f_1(X_j)\,f_{2a}(X_i)\,f_{2b}(X_k)$:

$$
T(X_j \to X_i)_F = E_i\Big[f_{2a}(X_i)\, E_k(f_{2b} \mid X_i)\, \frac{d}{dX_i} E_j(f_1 \mid X_i)\Big]
$$

$f_1$: driver part. $f_{2a}$: target part. $f_{2b}$: outer part.

Only terms with $X_j$ count (Eq. 27). Apply to each term and add (p. 12, p. 14).

Paper's example (Eq. 38a, 39a): $dX_1/dt = \exp(X_1)\sin(X_2) + X_1X_3$, so $f_1 = \sin X_2$, $f_{2a} = \exp X_1$:

$$
T(X_2 \to X_1)_F = E_1\Big[\exp(X_1)\,\frac{dE_2(\sin X_2 \mid X_1)}{dX_1}\Big]
$$

## Tool 3: `term37a(theta, f1, f2a, f2b)` (Eq. 37a, 50a)

One term $\hat\theta\, f_1(X_j)\, f_{2a}(X_2)\, f_{2b}(X_k)$:

$$
\hat\theta\; E_2\Big[f_{2a}(X_2)\; E(f_{2b} \mid X_2)\; \frac{dE(f_1 \mid X_2)}{dX_2}\Big]
$$

$\hat\theta$ out front; $f_{2a}$ as is; $f_{2b}$ through its curve; $f_1$ through the slope of its curve. Missing factor = 1. Add the terms (Eq. 49a).

In [12]:
def term37a(theta, f1, f2a=None, f2b=None):
    d = fit_curve(X2, f1)
    if f2b is not None:
        b = fit_curve(X2, f2b)
    total = 0.0
    for n in range(N):
        a = f2a[n] if f2a is not None else 1.0
        e = value(b, X2[n]) if f2b is not None else 1.0
        total += a * e * slope(d, X2[n])
    return theta * total / N

### $X_3 \to X_2$ (Eq. 72e)

Only $-\frac{s_1 s_3}{s_2}X_1X_3$ has $X_3$: $\theta = -\frac{s_1 s_3}{s_2}$, $f_1 = X_3$, $f_{2b} = X_1$.

$$
T(X_3 \to X_2)_F = -\frac{s_1 s_3}{s_2}\, E_2\Big[E_1(X_1 \mid X_2)\, \frac{d E_3(X_3 \mid X_2)}{dX_2}\Big]
$$

Paper: −0.73 (Table 2, CSM).

In [13]:
k = s1 * s3 / s2
print("T(X3 -> X2):", term37a(-k, X3, f2b=X1), "  (paper: -0.73)")

T(X3 -> X2): -0.7252369275164938   (paper: -0.73)


### $X_1 \to X_2$ (Eq. 72c)

Two terms have $X_1$:

| term | $\theta$ | $f_1$ | $f_{2a}$ | $f_{2b}$ |
|---|---|---|---|---|
| $\frac{s_1}{s_2}(R_a - \mu_3)X_1$ | $\frac{s_1}{s_2}(R_a - \mu_3)$ | $X_1$ | | |
| $-\frac{s_1 s_3}{s_2}X_1X_3$ | $-\frac{s_1 s_3}{s_2}$ | $X_1$ | | $X_3$ |

Same $f_1$, so in the paper the two merge into one bracket:

$$
T(X_1 \to X_2)_F = E_2\Big[\Big(\frac{s_1}{s_2}(R_a - \mu_3) - \frac{s_1 s_3}{s_2}E_3(X_3 \mid X_2)\Big)\frac{d E_1(X_1 \mid X_2)}{dX_2}\Big]
$$

Paper: 3.58 (Table 2, CSM).

In [14]:
a = (s1 / s2) * (Ra - mu3)
T = term37a(a, X1) + term37a(-k, X1, f2b=X3)
print("T(X1 -> X2):", T, "  (paper: 3.58)")

T(X1 -> X2): 3.5843165529698098   (paper: 3.58)
